**Notebook overview**
- Purpose: Scrape The War Zone for search keywords and convert scraped records into a standardized CSV for analysis.
- Produces: raw scraped JSON under `data/raw/` and cleaned CSV(s) under `data/processed/` with schema: `date`, `news source`, `title`, `link`.
- Notes: Processing parses dates, applies keyword filters, and deduplicates on `link`.



In [ ]:
import json
import re
import time
import urllib.parse
from bs4 import BeautifulSoup
import requests

# Scraper: loops keywords, fetches search results, and extracts article fields
def scrape_news_by_keywords(keywords):
    """Loops through a list of search keywords, extracts search results, cleans formatting, removes duplicate articles, and returns a structured JSON array."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) ProjectDataPipeline"
    }

    # Dictionary to prevent duplicate articles across different keyword searches
    unique_articles = {}

    print("==================================================")
    print(f"Starting news extraction for {len(keywords)} keywords...")
    print("==================================================")

    for keyword in keywords:
        encoded_query = urllib.parse.quote(keyword)
        url = f"https://www.twz.com/?s={encoded_query}"

        print(f"Searching: '{keyword}' -> {url}")

        try:
            response = requests.get(url, headers=headers, timeout=15)
            if response.status_code != 200:
                print(f"   Skipped: HTTP Status {response.status_code}")
                continue

            soup = BeautifulSoup(response.text, "html.parser")
            articles = soup.find_all("div", class_="post-content")
            new_items_found = 0

            for article in articles:
                # Internal helper to clean text fields
                def clean_text(element):
                    if not element:
                        return None
                    return re.sub(r"\s+", " ", element.text.strip())

                # Extract individual fields
                h3_tag = article.find("h3", class_="card-post-title")
                title = clean_text(h3_tag)

                date_tag = article.find("p", class_="byline-item-timestamp")
                date = clean_text(date_tag)
                if date:
                    date = (
                        date.replace("Posted on ", "")
                        .replace("Updated on ", "")
                        .strip()
                    )

                author_tag = article.find("a", class_="byline-link")
                author = clean_text(author_tag)

                category_tag = article.find("a", class_="cat-name-badge")
                category = clean_text(category_tag)

                link_tag = article.find(
                    "a", class_="card-post-title-link"
                ) or article.find("a")
                article_url = link_tag.get("href") if link_tag else None

                if article_url:
                    # Resolve relative paths to full URLs if necessary
                    if article_url.startswith("/"):
                        article_url = f"https://www.twz.com{article_url}"

                    # Add to dictionary if this article URL has not been processed yet
                    if article_url not in unique_articles:
                        unique_articles[article_url] = {
                            "title": title,
                            "date": date,
                            "author": author,
                            "category": category,
                            "url": article_url,
                        }
                        new_items_found += 1

            print(f"   Added {new_items_found} new unique articles.")
            time.sleep(1.5)

        except Exception as error:
            print(f"   Error processing keyword '{keyword}': {error}")

    # Convert the dictionary values into a direct list
    articles_list = list(unique_articles.values())

    # Format the final clean list as a JSON array string
    json_output = json.dumps(articles_list, indent=4)

    print("\n=========================================")
    print("EXTRACTION COMPLETE")
    print(f"Total Unique Articles Saved: {len(articles_list)}")
    print("=========================================")

    return json_output


In [13]:
from pathlib import Path
from datetime import datetime

# Define your search keywords directly
search_terms = [
    "ukraine missile", 
    "shahed drone", 
    "black sea fleet"
    ]

# Run the clean pipeline
scraped_data_json = scrape_news_by_keywords(search_terms)

# Preview the clean output string
if scraped_data_json:
    print(scraped_data_json[:1000])

    # Save raw output into News/data/raw/ with timestamped filename
    raw_dir = Path.cwd()
    if raw_dir.name.lower() != "news" and (raw_dir / "News").exists():
        raw_dir = raw_dir / "News"
    raw_dir = raw_dir / "data" / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)

    executed_at = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = raw_dir / f"warzone_raw_{executed_at}.json"
    output_path.write_text(scraped_data_json, encoding="utf-8")
    print(f"💾 Raw WarZone JSON saved: {output_path}")


Starting news extraction for 3 keywords...
Searching: 'ukraine missile' -> https://www.twz.com/?s=ukraine%20missile
   Added 22 new unique articles.
Searching: 'shahed drone' -> https://www.twz.com/?s=shahed%20drone
   Added 20 new unique articles.
Searching: 'black sea fleet' -> https://www.twz.com/?s=black%20sea%20fleet
   Added 20 new unique articles.

EXTRACTION COMPLETE
Total Unique Articles Saved: 62
[
    {
        "title": "No Long-Range Missiles For Ukraine: Trump",
        "date": "Jul 15, 2025",
        "author": "Howard Altman",
        "category": "Russia",
        "url": "https://www.twz.com/news-features/no-long-range-missiles-for-ukraine-trump"
    },
    {
        "title": "New Russian Air-Launched Cruise Missile Appears In Ukraine",
        "date": "Mar 3, 2026",
        "author": "Thomas Newdick",
        "category": "Russian Air Force",
        "url": "https://www.twz.com/air/new-russian-air-launched-cruise-missile-appears-in-ukraine"
    },
    {
        "title": "

In [16]:
# --- Process raw WarZone JSON -> standardized CSV(s) ---
import json
from pathlib import Path

import pandas as pd

ANALYSIS_KEYWORDS = [
    "ukraine",
    "missile",
    "explosion",
    "drone",
    "strike",
    "attack",
    "shelling",
    "bombardment",
    "blast",
    "black sea fleet",
    "blackout",
    "power outage",
]

RAW_DIR = Path("data") / "raw"
PROC_DIR = Path("data") / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)


def parse_datetime(value):
    if value is None or value == "":
        return pd.NaT
    text = str(value).strip()
    parsed = pd.to_datetime(text, errors="coerce")
    if pd.isna(parsed):
        return pd.NaT
    if not any(token in text.lower() for token in [":", "am", "pm", "t"]):
        parsed = parsed.normalize() + pd.Timedelta(hours=12)
    return parsed


def matches_keywords(text):
    lowered = (text or "").lower()
    return any(keyword in lowered for keyword in ANALYSIS_KEYWORDS)


for raw_file in sorted(RAW_DIR.glob("warzone*.json")):
    try:
        data = json.loads(raw_file.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Failed to load {raw_file}: {exc}")
        continue

    if isinstance(data, dict) and data.get("articles"):
        records = data.get("articles")
    elif isinstance(data, list):
        records = data
    else:
        records = []

    rows = []
    for record in records:
        title = record.get("title", "")
        link = record.get("url", "")
        if not title or not link:
            continue

        if not matches_keywords(f"{title} {record.get('category', '')} {record.get('author', '')}"):
            continue

        rows.append(
            {
                "date": parse_datetime(record.get("date", "")),
                "news source": "War Zone",
                "title": title,
                "link": link,
            }
        )

    if not rows:
        print(f"No matching records found in {raw_file}")
        continue

    df = pd.DataFrame(rows)
    df = df.dropna(subset=["date", "title", "link"])
    df = df.drop_duplicates(subset=["link"]).reset_index(drop=True)
    df = df[["date", "news source", "title", "link"]]

    processed_name = raw_file.name.replace("_raw", "")
    processed_path = PROC_DIR / processed_name.replace(".json", ".csv")
    df.to_csv(processed_path, index=False, encoding="utf-8")
    print(f"Saved processed WarZone to: {processed_path} ({len(df)} rows)")


Saved processed WarZone to: data\processed\warzone_20260530_141026.csv (60 rows)


**What this notebook does**
- Scrapes The War Zone search results for the selected keywords and saves the raw JSON under `data/raw/`.
- Re-processes each saved raw file into a standardized CSV under `data/processed/`.

**Cleaning process**
- Read the scraped article records from the raw JSON.
- Keep only the shared analysis columns: `date`, `news source`, `title`, `link`.
- Parse the scraped date into a full datetime and default the time to 12:00 when only a date is available.
- Set the news source label to `War Zone`.
- Apply keyword filtering again during processing so the final CSV keeps only relevant articles.
- Remove duplicate rows by `link` and save the cleaned CSV with `_raw` removed from the filename.

---
` Merging all news`


In [3]:
# Incremental merge for War Zone: keep history in warzone_all.csv and append newcomers
from pathlib import Path
import pandas as pd

PROC_DIR = Path("data") / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROC_DIR / "warzone_all.csv"

wz_files = sorted(
    f for f in PROC_DIR.glob("warzone*.csv")
    if f.name.lower() != "warzone_all.csv"
)

frames = []

if out_path.exists():
    try:
        existing_all = pd.read_csv(out_path)
        frames.append(existing_all)
        print(f"Loaded existing master {out_path.name}: {len(existing_all)} rows")
    except Exception as exc:
        print(f"Skipped existing master due to read error: {exc}")

for f in wz_files:
    try:
        df = pd.read_csv(f)
        frames.append(df)
        print(f"Loaded {f.name}: {len(df)} rows")
    except Exception as exc:
        print(f"Skipped {f.name} due to read error: {exc}")

if not frames:
    print(f"No War Zone data found in: {PROC_DIR.resolve()}")
else:
    merged = pd.concat(frames, ignore_index=True)

    preferred_cols = ["date", "news source", "title", "link"]
    existing_preferred = [c for c in preferred_cols if c in merged.columns]
    if existing_preferred:
        merged = merged[existing_preferred + [c for c in merged.columns if c not in existing_preferred]]

    if "link" in merged.columns:
        merged = merged.drop_duplicates(subset=["link"], keep="first")
    elif all(c in merged.columns for c in ["title", "date"]):
        merged = merged.drop_duplicates(subset=["title", "date"], keep="first")
    else:
        merged = merged.drop_duplicates()

    merged.to_csv(out_path, index=False, encoding="utf-8")
    print(f"Saved incremental War Zone master: {out_path.resolve()} ({len(merged)} rows)")

merged.head() if 'merged' in locals() else None

Loaded existing master warzone_all.csv: 60 rows
Loaded warzone_20260530_141026.csv: 60 rows
Saved incremental War Zone master: C:\.Okul\CE49X Introduction to Computational Thinking and Data Science for Civil Engineers\GithubRepository\CE49X\Final Project\News\data\processed\warzone_all.csv (60 rows)


,date,news source,title,link
0,2025-07-15 12:00:00,War Zone,No Long-Range Missiles For Ukraine: Trump,https://www.twz.com/news-features/no-long-rang...
1,2026-03-03 12:00:00,War Zone,New Russian Air-Launched Cruise Missile Appear...,https://www.twz.com/air/new-russian-air-launch...
2,2026-04-27 12:00:00,War Zone,Evidence Of Ukraine Using AIM-120C-8 Missiles ...,https://www.twz.com/air/evidence-of-ukraine-us...
3,2025-10-07 00:00:00,War Zone,New ‘Bulged’ Neptune Cruise Missile Variant Em...,https://www.twz.com/news-features/new-bulged-n...
4,2026-01-21 12:00:00,War Zone,Claims Swirl Around Use Of New Russian Missile...,https://www.twz.com/land/claims-swirl-around-u...
